In [1]:
# Import all pacakges
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error, r2_score
from sklearn.metrics import balanced_accuracy_score, precision_score, average_precision_score, roc_auc_score, recall_score, cohen_kappa_score, matthews_corrcoef, f1_score
import shap
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np


/Users/saraziadat/miniconda3/envs/bbb_exc_score_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# compare different datasets: 

all_outerfolds, all_fulls = [],[]

ds_list = ['ecfp', 'maccs', 'mordred', 'ecfp_maccs', 'ecfp_mordred', 'mordred_maccs', 'full_dataset']

for ds in ds_list:

    outerfold_df = pd.read_csv(f'../model_evaluation/{ds}/supp_figure_data_per_outerfold_all_data_15_{ds}.csv')
    full_df = pd.read_csv(f'../model_evaluation/{ds}/supp_figure_data_per_outerfold_test_data_only_15_{ds}.csv')

    outerfold_data = [f'{outerfold_df.iloc[5, i]:.3f} ± {outerfold_df.iloc[6, i]:.3f}' for i in range(1,len(list(outerfold_df.columns)))]
    full_data = [f'{full_df.iloc[5, i]:.3f} ± {full_df.iloc[6, i]:.3f}' for i in range(1,len(list(full_df.columns)))]
    
    all_outerfolds.append(outerfold_data)
    all_fulls.append(full_data)


outer_mean_std_df = pd.DataFrame(all_outerfolds, columns = outerfold_df.columns[1:], index=ds_list)
full_mean_std_df = pd.DataFrame(all_fulls, columns = full_df.columns[1:], index=ds_list)

outer_mean_std_df.to_csv('../model_evaluation/dataset_comparison_test_data_only.csv')
full_mean_std_df.to_csv('../model_evaluation/dataset_comparison_all_data.csv')

In [3]:
# Table 1: Classification performance on molecules that do not have any regression (logBB) labels. 
all_classification_results = []

ds_list = ['baseline','full_dataset']

literature_no_logbb = pd.read_csv('../model_evaluation/literature_comparison_classification.csv')
all_classification_results.append(literature_no_logbb.iloc[3, :].tolist()[1:])
all_classification_results.append(literature_no_logbb.iloc[2, :].tolist()[1:])

for ds in ds_list:

    outerfold_df = pd.read_csv(f'../model_evaluation/{ds}/supp_figure_data_per_outerfold_test_data_only_no_logbb_15_{ds}.csv')
    full_df = pd.read_csv(f'../model_evaluation/{ds}/supp_figure_data_per_outerfold_all_data_no_logbb_15_{ds}.csv')

    outerfold_data = [f'{outerfold_df.iloc[5, i]:.3f} ± {outerfold_df.iloc[6, i]:.3f}' for i in range(1,len(list(outerfold_df.columns)))]
    full_data = [f'{full_df.iloc[5, i]:.3f} ± {full_df.iloc[6, i]:.3f}' for i in range(1,len(list(full_df.columns)))]
    
    all_classification_results.append(outerfold_data)
    all_classification_results.append(full_data)

all_classification_results_df = pd.DataFrame(all_classification_results, columns = outerfold_df.columns[1:], index=['radchenko_pred_unlabelled', 'shaker_logbb_pred_unlabelled', 'baseline_test_data', 'baseline_all_data', 'our_model_test_data', 'our_model_all_data', ])
all_classification_results_df.to_csv('../model_evaluation/classification_no_logbb_results.csv')

all_classification_results_df

,balanced_acc,prec,recall,rocauc,prauc,kappa,MCC,f1
radchenko_pred_unlabelled,0.593777,0.616368,0.286564,0.593777,0.431298,0.212893,0.241524,0.391234
shaker_logbb_pred_unlabelled,0.535016,0.777778,0.083234,0.535016,0.391987,0.087392,0.175047,0.150376
baseline_test_data,0.500 ± 0.002,0.200 ± 0.400,0.001 ± 0.002,0.500 ± 0.002,0.386 ± 0.013,-0.000 ± 0.005,0.001 ± 0.036,0.002 ± 0.004
baseline_all_data,0.500 ± 0.000,0.200 ± 0.400,0.000 ± 0.000,0.500 ± 0.000,0.385 ± 0.000,-0.000 ± 0.001,0.001 ± 0.016,0.000 ± 0.001
our_model_test_data,0.565 ± 0.028,0.660 ± 0.113,0.196 ± 0.043,0.565 ± 0.028,0.441 ± 0.035,0.149 ± 0.064,0.200 ± 0.081,0.300 ± 0.059
our_model_all_data,0.571 ± 0.009,0.674 ± 0.023,0.204 ± 0.029,0.571 ± 0.009,0.444 ± 0.007,0.162 ± 0.018,0.215 ± 0.015,0.311 ± 0.033


In [4]:
# Table 2: Classification performance on all molecules labelled for their BBB Class. 
# Table 3: Regression performance on the full dataset. 

all_classification_results = []

ds_list = ['baseline','mordred']

literature_classification = pd.read_csv('../model_evaluation/literature_comparison_classification.csv')
literature_regression = pd.read_csv('../model_evaluation/literature_comparison_regression.csv')

all_classification_results.append((literature_regression.iloc[1, :].tolist()[1:]+literature_classification.iloc[1, :].tolist()[1:]))
all_classification_results.append((literature_regression.iloc[0, :].tolist()[1:]+literature_classification.iloc[0, :].tolist()[1:]))

for ds in ds_list:

    outerfold_df = pd.read_csv(f'../model_evaluation/{ds}/supp_figure_data_per_outerfold_test_data_only_15_{ds}.csv')
    full_df = pd.read_csv(f'../model_evaluation/{ds}/supp_figure_data_per_outerfold_all_data_15_{ds}.csv')

    outerfold_data = [f'{outerfold_df.iloc[5, i]:.3f} ± {outerfold_df.iloc[6, i]:.3f}' for i in range(1,len(list(outerfold_df.columns)))]
    full_data = [f'{full_df.iloc[5, i]:.3f} ± {full_df.iloc[6, i]:.3f}' for i in range(1,len(list(full_df.columns)))]
    
    all_classification_results.append(outerfold_data)
    all_classification_results.append(full_data)

all_classification_results_df = pd.DataFrame(all_classification_results, columns = outerfold_df.columns[1:], index=['radchenko_pred', 'shaker_logbb_pred','baseline_test_data', 'baseline_all_data', 'our_model_test_data', 'our_model_all_data'])


all_classification_results_df.to_csv('../model_evaluation/regression_classification_all_results.csv')